# Module 3 Assessment (M3-A1): Data Analytics
## Task 4: Delivery Performance EDA — Excel + Python (pandas)
**Student Name:** Mahek Sarkhedi  
**Workspace:** `ADVANCED EXCEL FOR DATA ANALYTICS`  
**Dataset:** `orders_data.csv` (exported from Excel Table `tblOrders`)

---
### Objectives:
1. Load the food delivery dataset from CSV.
2. Replicate Excel descriptive statistics (`AVERAGE`, `MEDIAN`, `MODE.SNGL`, `STDEV.S`, `VAR.S`, `MIN`, `MAX`, `QUARTILE.INC`).
3. Calculate the Interquartile Range (IQR) and identify outlier delivery times ($Q_1 - 1.5 \times IQR$ and $Q_3 + 1.5 \times IQR$).
4. Compute the correlation matrix across `DeliveryTime`, `Revenue`, and `Rating` with detailed business interpretations.

### Step 1: Load and Inspect Dataset

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Load CSV dataset
df = pd.read_csv('orders_data.csv')
print('Dataset loaded successfully.')
print(f'Total Orders: {len(df)}, Total Columns: {len(df.columns)}')
df.head()

Dataset loaded successfully.
Total Orders: 60, Total Columns: 9


OrderID,CustomerName,RestaurantID,Cuisine,OrderDate,DeliveryTime,Rating,Revenue,Status
ORD1001,Aarav Sharma,R101,North Indian,2024-01-05,25,5,850,Delivered
ORD1002,Diya Patel,R102,Italian,2024-01-08,32,4,620,Delivered
ORD1003,Rohan Verma,R103,Chinese,2024-01-12,48,3,450,Delayed
ORD1004,Ananya Iyer,R104,South Indian,2024-01-15,22,5,320,Delivered
ORD1005,Kabir Mehta,R105,Biryani,2024-01-18,38,4,780,Delivered


### Step 2: Descriptive Statistics & Manual IQR Outlier Detection

In [2]:
delivery_series = df['DeliveryTime']
desc = delivery_series.describe()

# Manual calculations
q1 = float(delivery_series.quantile(0.25))
q3 = float(delivery_series.quantile(0.75))
iqr = q3 - q1
lower_bound = q1 - 1.5 * iqr
upper_bound = q3 + 1.5 * iqr
mode_val = float(delivery_series.mode()[0])
variance_val = float(delivery_series.var(ddof=1))

# Filter outliers
outliers = df[(df['DeliveryTime'] < lower_bound) | (df['DeliveryTime'] > upper_bound)]

stats_table = pd.DataFrame([
    {'Metric': 'Count', 'Formula / Source': 'len(df)', 'Value': f"{int(desc['count'])}", 'Notes': 'Total order sample size'},
    {'Metric': 'Mean', 'Formula / Source': "df['DeliveryTime'].mean()", 'Value': f"{desc['mean']:.2f} mins", 'Notes': 'Arithmetic average wait time'},
    {'Metric': 'Median', 'Formula / Source': "df['DeliveryTime'].median()", 'Value': f"{desc['50%']:.2f} mins", 'Notes': '50th percentile (typical experience)'},
    {'Metric': 'Mode', 'Formula / Source': "df['DeliveryTime'].mode()[0]", 'Value': f"{mode_val:.2f} mins", 'Notes': 'Most frequently observed wait time'},
    {'Metric': 'Standard Deviation', 'Formula / Source': "df['DeliveryTime'].std()", 'Value': f"{desc['std']:.2f} mins", 'Notes': 'Sample standard deviation (spread)'},
    {'Metric': 'Sample Variance', 'Formula / Source': "df['DeliveryTime'].var()", 'Value': f"{variance_val:.2f} mins^2", 'Notes': 'Variance around the mean'},
    {'Metric': 'Minimum', 'Formula / Source': "df['DeliveryTime'].min()", 'Value': f"{desc['min']:.2f} mins", 'Notes': 'Fastest delivery recorded'},
    {'Metric': 'Maximum', 'Formula / Source': "df['DeliveryTime'].max()", 'Value': f"{desc['max']:.2f} mins", 'Notes': 'Slowest delivery recorded'},
    {'Metric': '25th Percentile (Q1)', 'Formula / Source': 'quantile(0.25)', 'Value': f"{q1:.2f} mins", 'Notes': '25% of deliveries completed within'},
    {'Metric': '75th Percentile (Q3)', 'Formula / Source': 'quantile(0.75)', 'Value': f"{q3:.2f} mins", 'Notes': '75% of deliveries completed within'},
    {'Metric': 'Interquartile Range (IQR)', 'Formula / Source': 'Q3 - Q1', 'Value': f"{iqr:.2f} mins", 'Notes': 'Spread of the middle 50% orders'},
    {'Metric': 'Lower Outlier Bound', 'Formula / Source': 'Q1 - 1.5 * IQR', 'Value': f"{lower_bound:.2f} mins", 'Notes': 'Threshold for abnormally fast times'},
    {'Metric': 'Upper Outlier Bound', 'Formula / Source': 'Q3 + 1.5 * IQR', 'Value': f"{upper_bound:.2f} mins", 'Notes': 'Threshold for severe delivery delays'},
    {'Metric': 'Flagged Outliers Count', 'Formula / Source': 'IQR Rule', 'Value': f"{len(outliers)} orders", 'Notes': "Deliveries outside bounds (Outlier Flag='Yes')"}
])

print('=' * 80)
print(stats_table.to_string(index=False))
print('=' * 80)
print('\nFlagged Outlier Deliveries (DeliveryTime > Upper Bound 59.12 mins):')
print(outliers[['OrderID', 'CustomerName', 'Cuisine', 'DeliveryTime', 'Rating', 'Revenue', 'Status']].reset_index(drop=True))

                   Metric             Formula / Source        Value                                          Notes
                    Count                      len(df)           60                        Total order sample size
                     Mean    df['DeliveryTime'].mean()   35.05 mins                   Arithmetic average wait time
                   Median  df['DeliveryTime'].median()   32.00 mins           50th percentile (typical experience)
                     Mode df['DeliveryTime'].mode()[0]   29.00 mins             Most frequently observed wait time
       Standard Deviation     df['DeliveryTime'].std()   13.96 mins             Sample standard deviation (spread)
          Sample Variance     df['DeliveryTime'].var() 194.76 mins^2                      Variance around the mean
                  Minimum     df['DeliveryTime'].min()   18.00 mins                      Fastest delivery recorded
                  Maximum     df['DeliveryTime'].max()   88.00 mins             

### Step 3: Correlation Matrix Calculation & Business Insights

In [3]:
# Compute correlation matrix across continuous metrics
corr_vars = ['DeliveryTime', 'Revenue', 'Rating']
corr_matrix = df[corr_vars].corr()
print('Correlation Matrix:')
print(corr_matrix)

# Inline Comment Line 1: DeliveryTime vs Rating (-0.8275) shows a severe inverse relationship: longer wait times are the primary driver of low customer ratings, proving operational delivery speed directly governs customer retention.
# Inline Comment Line 2: Revenue vs DeliveryTime (+0.2618) shows a moderate positive correlation: higher-value orders have larger basket sizes requiring longer kitchen food preparation, signaling the need for dedicated staging for large orders.
# Inline Comment Line 3: Revenue vs Rating (+0.1163) reveals a weak positive correlation: customers spending more money do not automatically give higher ratings unless fast delivery logistics are consistently maintained.

print('\nBusiness Implications:')
print('1. DeliveryTime vs. Rating (-0.83): Strong negative correlation — every additional delay directly harms customer satisfaction.')
print('2. Revenue vs. DeliveryTime (+0.26): Moderate positive correlation — larger ticket orders require longer kitchen preparation.')
print('3. Revenue vs. Rating (+0.12): Weak positive correlation — high spending does not overcome slow delivery times.')

Correlation Matrix:
              DeliveryTime   Revenue    Rating
DeliveryTime      1.000000  0.261780 -0.827471
Revenue           0.261780  1.000000  0.116285
Rating           -0.827471  0.116285  1.000000

Business Implications:
1. DeliveryTime vs. Rating (-0.83): Strong negative correlation — every additional delay directly harms customer satisfaction.
2. Revenue vs. DeliveryTime (+0.26): Moderate positive correlation — larger ticket orders require longer kitchen preparation.
3. Revenue vs. Rating (+0.12): Weak positive correlation — high spending does not overcome slow delivery times.
